In [1]:
"""
Download NOAA CRW 5km daily data for the Great Barrier Reef bounding box,
then aggregate to weekly resolution to match CoRTAD training format.
 
Downloads from PacIOOS ERDDAP (coastwatch.pfeg.noaa.gov) which hosts
the NOAA_DHW dataset containing all CRW variables in one place.
 
CRW variable -> CoRTAD training equivalent:
  CRW_SST         -> FilledSST
  CRW_HOTSPOT     -> TSA (clipped to >=0)
  CRW_DHW         -> TSA_DHW
  CRW_BAA         -> (extra, not used in model but useful for visualization)
  TSA_Frequency   -> computed from CRW_HOTSPOT (not a CRW product)
 
Output: gbr_crw_weekly.nc — weekly aggregated data ready for model inference
"""
 
import requests
import pandas as pd
import numpy as np
import xarray as xr
from io import BytesIO
from datetime import datetime, timedelta
import os
import time as time_module

In [4]:
 
# ──────────────────────────────────────────────────────────────
# CONFIG
# ──────────────────────────────────────────────────────────────
 
# GBR bounding box
LAT_MIN = -24.5
LAT_MAX = -10.0
LON_MIN = 142.0
LON_MAX = 154.0
 
# ERDDAP uses 0-360 longitude for NOAA_DHW dataset
LON_MIN_360 = LON_MIN        # 142.0 (already positive, no conversion needed)
LON_MAX_360 = LON_MAX        # 154.0
 
# Time range: 365 days back from a fixed point
# Adjust this date as needed
END_DATE = "2026-03-10"
START_DATE = (pd.Timestamp(END_DATE) - pd.Timedelta(days=365)).strftime("%Y-%m-%d")
 
# ERDDAP base URL
ERDDAP_BASE = "https://coastwatch.pfeg.noaa.gov/erddap/griddap/NOAA_DHW"
 
# Variables to download
# CRW_SST: Sea Surface Temperature (CoralTemp)
# CRW_SSTANOMALY: SST Anomaly
# CRW_HOTSPOT: Coral Bleaching HotSpot (≈ TSA clipped to >=0)
# CRW_DHW: Degree Heating Weeks
# CRW_BAA: Bleaching Alert Area (0-5 scale)
VARIABLES = ["CRW_SST", "CRW_SSTANOMALY", "CRW_HOTSPOT", "CRW_DHW", "CRW_BAA"]
 
OUTPUT_DAILY = "gbr_crw_daily.nc"
OUTPUT_WEEKLY = "gbr_crw_weekly.nc"

In [5]:
# ──────────────────────────────────────────────────────────────
# STEP 1: Download daily data in monthly chunks
# ──────────────────────────────────────────────────────────────
print("=" * 70)
print("STEP 1: Downloading CRW daily data from ERDDAP")
print("=" * 70)
print(f"  Bounding box: lat [{LAT_MIN}, {LAT_MAX}], lon [{LON_MIN}, {LON_MAX}]")
print(f"  Date range: {START_DATE} to {END_DATE}")
print(f"  Variables: {VARIABLES}")
 
# Split into monthly chunks to avoid server timeouts
start = pd.Timestamp(START_DATE)
end = pd.Timestamp(END_DATE)
 
chunks = []
chunk_start = start
while chunk_start < end:
    chunk_end = min(chunk_start + pd.DateOffset(months=1) - pd.Timedelta(days=1), end)
    chunks.append((chunk_start, chunk_end))
    chunk_start = chunk_end + pd.Timedelta(days=1)
 
print(f"  Downloading in {len(chunks)} monthly chunks\n")
 
all_datasets = []
 
for i, (cs, ce) in enumerate(chunks):
    cs_str = cs.strftime("%Y-%m-%dT12:00:00Z")
    ce_str = ce.strftime("%Y-%m-%dT12:00:00Z")
    
    # Build ERDDAP query for NetCDF subset
    var_query = ",".join([
        f"{v}[({cs_str}):1:({ce_str})][({LAT_MIN}):1:({LAT_MAX})][({LON_MIN_360}):1:({LON_MAX_360})]"
        for v in VARIABLES
    ])
    url = f"{ERDDAP_BASE}.nc?{var_query}"
    
    print(f"  Chunk {i+1}/{len(chunks)}: {cs.strftime('%Y-%m-%d')} to {ce.strftime('%Y-%m-%d')}")
    print(f"    URL: {url[:120]}...")
    
    retries = 3
    for attempt in range(retries):
        try:
            t0 = time_module.time()
            resp = requests.get(url, timeout=300)
            resp.raise_for_status()
            
            ds = xr.open_dataset(BytesIO(resp.content))
            all_datasets.append(ds)
            
            elapsed = time_module.time() - t0
            size_mb = len(resp.content) / (1024 * 1024)
            print(f"    OK: {size_mb:.1f} MB in {elapsed:.1f}s")
            break
            
        except Exception as e:
            print(f"    Attempt {attempt+1}/{retries} FAILED: {type(e).__name__}: {e}")
            if attempt < retries - 1:
                wait = 10 * (attempt + 1)
                print(f"    Retrying in {wait}s...")
                time_module.sleep(wait)
            else:
                print(f"    GIVING UP on this chunk")
    
    time_module.sleep(2)  # Be polite to the server
 
if not all_datasets:
    print("\nERROR: No data downloaded!")
    exit(1)
 
# Combine all chunks
print(f"\n  Combining {len(all_datasets)} chunks...")
daily = xr.concat(all_datasets, dim="time")
daily = daily.sortby("time")
 
# Convert longitude back to -180 to 180 if needed
if daily.longitude.values.max() > 180:
    daily = daily.assign_coords(longitude=(daily.longitude + 180) % 360 - 180)
    daily = daily.sortby("longitude")
 
print(f"  Combined shape: {dict(daily.dims)}")
print(f"  Time: {daily.time.values[0]} to {daily.time.values[-1]}")
print(f"  Lat: {daily.latitude.values.min():.2f} to {daily.latitude.values.max():.2f}")
print(f"  Lon: {daily.longitude.values.min():.2f} to {daily.longitude.values.max():.2f}")
 
# Save daily
daily.to_netcdf(OUTPUT_DAILY)
daily_size = os.path.getsize(OUTPUT_DAILY) / (1024 * 1024)
print(f"\n  Saved daily data: {OUTPUT_DAILY} ({daily_size:.1f} MB)")

STEP 1: Downloading CRW daily data from ERDDAP
  Bounding box: lat [-24.5, -10.0], lon [142.0, 154.0]
  Date range: 2025-03-10 to 2026-03-10
  Variables: ['CRW_SST', 'CRW_SSTANOMALY', 'CRW_HOTSPOT', 'CRW_DHW', 'CRW_BAA']

  Chunk 1/12: 2025-03-10 to 2025-04-09
    URL: https://coastwatch.pfeg.noaa.gov/erddap/griddap/NOAA_DHW.nc?CRW_SST[(2025-03-10T12:00:00Z):1:(2025-04-09T12:00:00Z)][(-2...
    Attempt 1/3 FAILED: ConnectionError: ('Connection aborted.', ConnectionResetError(54, 'Connection reset by peer'))
    Retrying in 10s...
    OK: 68.4 MB in 49.7s
  Chunk 2/12: 2025-04-10 to 2025-05-09
    URL: https://coastwatch.pfeg.noaa.gov/erddap/griddap/NOAA_DHW.nc?CRW_SST[(2025-04-10T12:00:00Z):1:(2025-05-09T12:00:00Z)][(-2...
    OK: 66.2 MB in 48.8s
  Chunk 3/12: 2025-05-10 to 2025-06-09
    URL: https://coastwatch.pfeg.noaa.gov/erddap/griddap/NOAA_DHW.nc?CRW_SST[(2025-05-10T12:00:00Z):1:(2025-06-09T12:00:00Z)][(-2...
    OK: 68.4 MB in 44.2s
  Chunk 4/12: 2025-06-10 to 2025-07-09
    UR

/var/folders/1x/6tp2zz4x70q7nl89j3cmqf7c0000gn/T/ipykernel_36074/578263049.py:80: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  print(f"  Combined shape: {dict(daily.dims)}")


  Combined shape: {'time': 245, 'latitude': 291, 'longitude': 241}
  Time: 2025-03-10T12:00:00.000000000 to 2025-12-09T12:00:00.000000000
  Lat: -24.48 to -9.98
  Lon: 142.02 to 154.02

  Saved daily data: gbr_crw_daily.nc (540.8 MB)


In [ ]:
# ──────────────────────────────────────────────────────────────
# STEP 2: Aggregate to weekly resolution
# ──────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("STEP 2: Aggregating to weekly resolution")
print("=" * 70)
 
# Resample to 7-day means (matching CoRTAD weekly convention)
weekly_mean = daily[["CRW_SST", "CRW_SSTANOMALY", "CRW_HOTSPOT"]].resample(time="7D").mean()
weekly_last = daily[["CRW_DHW"]].resample(time="7D").last()  # DHW is cumulative
weekly_max = daily[["CRW_BAA"]].resample(time="7D").max()    # Alert: take worst of the week
 
weekly = xr.merge([weekly_mean, weekly_last, weekly_max])
 
print(f"  Weekly shape: {dict(weekly.dims)}")
print(f"  {len(weekly.time)} weeks")

In [ ]:
# ──────────────────────────────────────────────────────────────
# STEP 3: Compute TSA_Frequency (not a native CRW product)
# ──────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("STEP 3: Computing TSA_Frequency from HotSpot data")
print("=" * 70)
 
# TSA_Frequency = count of weeks in trailing 52-week window where HotSpot >= 1
# Since we only have ~52 weeks of data, compute over all available weeks
hotspot = weekly["CRW_HOTSPOT"].values  # (time, lat, lon)
n_weeks = hotspot.shape[0]
 
freq = np.zeros_like(hotspot)
for t in range(n_weeks):
    window_start = max(0, t - 51)
    window = hotspot[window_start:t+1, :, :]
    freq[t] = np.nansum(window >= 1, axis=0)
 
weekly["TSA_Frequency"] = (("time", "latitude", "longitude"), freq)
print(f"  TSA_Frequency computed: shape {freq.shape}")

In [ ]:
# ──────────────────────────────────────────────────────────────
# STEP 4: Rename to match CoRTAD training feature names
# ──────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("STEP 4: Renaming variables to match training features")
print("=" * 70)
 
# Model was trained on: FilledSST, TSA, TSA_DHW, TSA_Frequency
weekly_model = weekly.rename({
    "CRW_SST": "FilledSST",
    "CRW_HOTSPOT": "TSA",        # HotSpot ≈ TSA (already clipped to >=0)
    "CRW_DHW": "TSA_DHW",
})
 
# Keep the extra variables for analysis but mark them clearly
weekly_model = weekly_model.rename({
    "CRW_SSTANOMALY": "SST_Anomaly_extra",
    "CRW_BAA": "Bleaching_Alert_extra",
})
 
print("  Variable mapping:")
print("    CRW_SST         -> FilledSST         (model feature)")
print("    CRW_HOTSPOT     -> TSA               (model feature)")
print("    CRW_DHW         -> TSA_DHW           (model feature)")
print("    TSA_Frequency   -> TSA_Frequency     (model feature, computed)")
print("    CRW_SSTANOMALY  -> SST_Anomaly_extra (for analysis)")
print("    CRW_BAA         -> Bleaching_Alert_extra (for analysis)")
 
# Save
weekly_model.to_netcdf(OUTPUT_WEEKLY)
weekly_size = os.path.getsize(OUTPUT_WEEKLY) / (1024 * 1024)
print(f"\n  Saved weekly data: {OUTPUT_WEEKLY} ({weekly_size:.1f} MB)")
 
# ──────────────────────────────────────────────────────────────
# SUMMARY
# ──────────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("SUMMARY")
print("=" * 70)
print(f"  Daily file:  {OUTPUT_DAILY} ({daily_size:.1f} MB)")
print(f"  Weekly file: {OUTPUT_WEEKLY} ({weekly_size:.1f} MB)")
print(f"  Grid: {len(weekly_model.latitude)} x {len(weekly_model.longitude)} pixels")
print(f"  Weeks: {len(weekly_model.time)}")
print(f"  Model features: FilledSST, TSA, TSA_DHW, TSA_Frequency")
print(f"\n  To load for inference:")
print(f"    ds = xr.open_dataset('{OUTPUT_WEEKLY}')")
print(f"    # Extract 16-week sequence at a reef point:")
print(f"    seq = ds[['FilledSST','TSA','TSA_DHW','TSA_Frequency']].sel(")
print(f"        latitude=-16.4, longitude=145.6, method='nearest'")
print(f"    ).isel(time=slice(-16, None)).to_array().values.T")
print(f"    # seq shape: (16, 4) — ready for model input")
 
print("\nDone!")